# Sentiment Analysis with LSTM and NLP

This notebook demonstrates sentiment analysis using Long Short-Term Memory (LSTM) neural networks combined with Natural Language Processing (NLP) techniques.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('sentiment_data.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

## 2. Data Analysis

In [ ]:
# Sentiment distribution
sentiment_counts = df['sentiment'].value_counts()
print("Sentiment Distribution:")
print(sentiment_counts)

# Visualize
plt.figure(figsize=(10, 5))
sentiment_counts.plot(kind='bar', color=['green', 'red'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Review length analysis
df['review_length'] = df['review'].apply(lambda x: len(x.split()))

print(f"Average review length: {df['review_length'].mean():.2f} words")
print(f"Max review length: {df['review_length'].max()} words")
print(f"Min review length: {df['review_length'].min()} words")

# Visualize
plt.figure(figsize=(10, 5))
plt.hist(df['review_length'], bins=20, color='steelblue', edgecolor='black')
plt.title('Distribution of Review Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. Text Preprocessing

In [ ]:
from sentiment_analysis import SentimentAnalysisLSTM

# Initialize model
model = SentimentAnalysisLSTM(vocab_size=5000, max_length=150, embedding_dim=100)

# Show preprocessing example
sample_text = df['review'].iloc[0]
print("Original text:")
print(sample_text)
print("\nCleaned text:")
print(model.preprocess_text(sample_text))

## 4. Model Training

In [ ]:
# Prepare data
X, y = model.prepare_data(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Build model
model.build_model()
model.model.summary()

In [ ]:
# Train model
history = model.train(X_train, y_train, X_val, y_val, epochs=10, batch_size=32)

## 5. Model Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Sentiment Prediction

In [ ]:
# Test predictions on new reviews
test_reviews = [
    "This product is absolutely amazing! I love it.",
    "Terrible quality. Very disappointed with my purchase.",
    "It's okay, nothing special but does the job.",
    "Best purchase ever! Highly recommended.",
    "Waste of money. Do not buy!"
]

print("\n" + "="*70)
print("SENTIMENT PREDICTIONS")
print("="*70)

for review in test_reviews:
    sentiment, confidence = model.predict_sentiment(review)
    print(f"\nReview: {review}")
    print(f"Sentiment: {sentiment.upper()} (Confidence: {confidence:.2%})")
    print("-" * 70)